# Person + Head Detection — Train & Test (Kazim)

Three detectors on the **Person Detection with head** dataset (Roboflow Universe, v2 YOLOv8 export,
classes `Head` / `Person`), all scored on the **same** metric so the comparison is honest:

| Track | Model | Approach |
|---|---|---|
| **A** | YOLO (Ultralytics) | pretrained COCO checkpoint, fine-tuned |
| **B** | RF-DETR Nano | pretrained transformer detector, fine-tuned |
| **C** | `ScratchDetector` | small YOLOv1-lite CNN, trained from scratch |

### What is different here vs `notebooks/Object_Detection_3_Tracks.ipynb`

That notebook trains all three tracks but scores each with a *different* tool — Ultralytics'
validator for A, `supervision.metrics` for B, and precision/recall/F1 at one fixed confidence
threshold for C. Those numbers cannot be put in the same table.

This notebook adds a **single shared VOC/COCO-style mAP implementation** (`Kazim/src/metrics.py`,
which has its own self-checks) and runs **every** track through it on the **test** split. It also
adds dataset EDA, per-class AP, qualitative side-by-side predictions, and a saved `results.json`.

**Before running:** enable a GPU — Colab `Runtime > Change runtime type > T4`, or
Kaggle `Settings > Accelerator > GPU T4 x2` **and `Internet: On`**.

Run **SETUP** first. After that the three track sections are independent — run them in any order.

In [1]:
# ============================================================
# SETUP — run this first
# ============================================================
import os
import sys
import json
import glob
import random
from pathlib import Path

# ---- Knobs -------------------------------------------------
# QUICK_MODE trims epochs and eval-set size so you can prove the whole notebook
# runs end to end in a few minutes. Set False for the real, reportable numbers.
QUICK_MODE = True

CLASS_NAMES = ["Head", "Person"]
NUM_CLASSES = len(CLASS_NAMES)
SEED = 42

random.seed(SEED)

# ---- Detect platform ---------------------------------------
if "google.colab" in sys.modules:
    PLATFORM = "colab"
elif os.path.exists("/kaggle/input"):
    PLATFORM = "kaggle"
else:
    PLATFORM = "local"
print("platform:", PLATFORM)

platform: local


In [2]:
# ============================================================
# SETUP (2) — locate the dataset
# ============================================================
# The dataset is the Roboflow "Person Detection with head" v2 YOLOv8 export.
# Point DATASET_ROOT at the folder that directly contains data.yaml.

def find_dataset_root(candidates):
    """Return the first candidate that actually holds a data.yaml, else None."""
    for c in candidates:
        for hit in sorted(glob.glob(c)):
            if os.path.exists(os.path.join(hit, "data.yaml")):
                return hit
    return None


if PLATFORM == "colab":
    # Upload the Roboflow zip to Drive, then let this cell unzip it.
    from google.colab import drive
    drive.mount("/content/drive")

    DATASET_ZIP = "/content/drive/MyDrive/Person Detection with head.v2i.yolov8.zip"  # <-- edit
    DATASET_ROOT = "/content/dataset"
    OUTPUT_ROOT = "/content/outputs"

    if not os.path.exists(os.path.join(DATASET_ROOT, "data.yaml")):
        import zipfile
        os.makedirs(DATASET_ROOT, exist_ok=True)
        print(f"unzipping {DATASET_ZIP} ...")
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(DATASET_ROOT)
        # Roboflow zips sometimes nest one folder deep.
        nested = find_dataset_root([DATASET_ROOT, os.path.join(DATASET_ROOT, "*")])
        if nested:
            DATASET_ROOT = nested

elif PLATFORM == "kaggle":
    OUTPUT_ROOT = "/kaggle/working/outputs"
    DATASET_ROOT = find_dataset_root(["/kaggle/input/*", "/kaggle/input/*/*"])

else:  # local
    _here = Path.cwd()
    _repo = next((p for p in [_here, *_here.parents] if (p / "src" / "common").exists()), _here)
    OUTPUT_ROOT = str(_repo / "Kazim" / "outputs")
    DATASET_ROOT = find_dataset_root([
        str(_repo / "Person Detection with head.v2i.yolov8"),
        str(_repo / "*"),
        str(_repo / "Kazim" / "*"),
    ])

assert DATASET_ROOT, (
    "Could not find data.yaml.\n"
    "Download the v2 YOLOv8 export from "
    "https://universe.roboflow.com/person-xf2dz/person-detection-with-head-8aw41/dataset/2 "
    "and set DATASET_ROOT to the unzipped folder."
)

os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATA_YAML = os.path.join(DATASET_ROOT, "data.yaml")
TRAIN_IMAGES = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_LABELS = os.path.join(DATASET_ROOT, "train", "labels")
VALID_IMAGES = os.path.join(DATASET_ROOT, "valid", "images")
VALID_LABELS = os.path.join(DATASET_ROOT, "valid", "labels")

# Some Roboflow exports ship no test split; fall back to valid so nothing crashes.
TEST_IMAGES = os.path.join(DATASET_ROOT, "test", "images")
TEST_LABELS = os.path.join(DATASET_ROOT, "test", "labels")
if not os.path.isdir(TEST_IMAGES):
    print("no test split found -> evaluating on valid split instead")
    TEST_IMAGES, TEST_LABELS = VALID_IMAGES, VALID_LABELS

print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT :", OUTPUT_ROOT)

DATASET_ROOT: /Users/ks/Desktop/Deep Learning 2/ObjectDetection_MathAdvance-main/Person Detection with head.v2i.yolov8
OUTPUT_ROOT : /Users/ks/Desktop/Deep Learning 2/ObjectDetection_MathAdvance-main/Kazim/outputs


In [3]:
# ============================================================
# SETUP (3) — dependencies and device
# ============================================================
# Colab/Kaggle already ship torch+CUDA, so only the detector libs are installed.
if PLATFORM in ("colab", "kaggle"):
    !pip install -q ultralytics "rfdetr[train,loggers]" supervision

import numpy as np
import torch
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")  # Apple Silicon
else:
    DEVICE = torch.device("cpu")

torch.manual_seed(SEED)

print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif DEVICE.type == "cpu":
    print("WARNING: no GPU. Training will be very slow — enable a GPU accelerator.")

ModuleNotFoundError: No module named 'torch'

In [4]:
# ============================================================
# SETUP (4) — shared mAP metric (Kazim/src/metrics.py)
# ============================================================
# All three tracks are scored with this one implementation, so the final table
# compares like with like. Falls back to an inline fetch-free copy check first.

_metrics_dir = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / "Kazim" / "src" / "metrics.py").exists():
        _metrics_dir = _cand / "Kazim" / "src"
        break
    if (_cand / "src" / "metrics.py").exists() and _cand.name == "Kazim":
        _metrics_dir = _cand / "src"
        break

assert _metrics_dir, (
    "metrics.py not found. Keep this notebook inside the repo (Kazim/src/metrics.py "
    "must be reachable), or upload metrics.py next to the notebook."
)

sys.path.insert(0, str(_metrics_dir))
from metrics import evaluate_detections, yolo_labels_to_xyxy, iou_matrix  # noqa: E402

print("loaded shared metric from:", _metrics_dir)

# Sanity: a perfect prediction must score exactly 1.0.
_gt = [{"boxes": np.array([[10, 10, 50, 50]]), "labels": np.array([0])}]
_pr = [{"boxes": np.array([[10, 10, 50, 50]]), "scores": np.array([0.9]), "labels": np.array([0])}]
assert abs(evaluate_detections(_pr, _gt)["mAP50"] - 1.0) < 1e-6
print("metric self-check OK")

loaded shared metric from: /Users/ks/Desktop/Deep Learning 2/ObjectDetection_MathAdvance-main/Kazim/src
metric self-check OK


In [5]:
# ============================================================
# Shared helpers — image/label pairing, used by every track
# ============================================================
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png")


def list_image_label_pairs(images_dir, labels_dir):
    """Pair each image with its YOLO .txt label. Images without a label file are
    skipped — an unlabelled image is not the same as a negative example."""
    images_dir, labels_dir = Path(images_dir), Path(labels_dir)
    pairs = []
    for ext in IMG_EXTS:
        for img_path in sorted(images_dir.glob(ext)):
            label_path = labels_dir / (img_path.stem + ".txt")
            if label_path.exists():
                pairs.append((img_path, label_path))
    return sorted(pairs)


def read_yolo_labels(label_path):
    """-> list of (cls_id, xc, yc, w, h), all normalised to [0,1]."""
    boxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                continue
            boxes.append((int(parts[0]), *map(float, parts[1:])))
    return boxes


TRAIN_PAIRS = list_image_label_pairs(TRAIN_IMAGES, TRAIN_LABELS)
VALID_PAIRS = list_image_label_pairs(VALID_IMAGES, VALID_LABELS)
TEST_PAIRS = list_image_label_pairs(TEST_IMAGES, TEST_LABELS)

print(f"train: {len(TRAIN_PAIRS):>6} labelled images")
print(f"valid: {len(VALID_PAIRS):>6} labelled images")
print(f"test : {len(TEST_PAIRS):>6} labelled images")

# How many test images each track is scored on (full split unless QUICK_MODE).
EVAL_N = 100 if QUICK_MODE else len(TEST_PAIRS)
EVAL_PAIRS = TEST_PAIRS[:EVAL_N]
print(f"\nscoring every track on the same {len(EVAL_PAIRS)} test images")

train:  12231 labelled images
valid:    899 labelled images
test :    846 labelled images

scoring every track on the same 100 test images


## 1. Dataset exploration

Before training anything, check what the data actually looks like: class balance, how many objects
per image, and how large the boxes are. Box size matters a lot here — `Head` boxes are small, and
small objects are exactly where the 128×128 from-scratch model (Track C) will struggle.

In [6]:
# ============================================================
# EDA — class balance and box geometry
# ============================================================
from collections import Counter

def split_stats(pairs, name):
    class_counts = Counter()
    boxes_per_image, areas = [], []
    for _, label_path in pairs:
        rows = read_yolo_labels(label_path)
        boxes_per_image.append(len(rows))
        for cls_id, _, _, w, h in rows:
            class_counts[cls_id] += 1
            areas.append((cls_id, w * h))
    return {
        "split": name,
        "images": len(pairs),
        "objects": sum(class_counts.values()),
        "class_counts": class_counts,
        "boxes_per_image": boxes_per_image,
        "areas": areas,
    }


stats = [split_stats(p, n) for p, n in
         [(TRAIN_PAIRS, "train"), (VALID_PAIRS, "valid"), (TEST_PAIRS, "test")]]

print(f"{'split':<8}{'images':>8}{'objects':>10}{'Head':>10}{'Person':>10}{'obj/img':>10}")
print("-" * 56)
for s in stats:
    n_head, n_person = s["class_counts"].get(0, 0), s["class_counts"].get(1, 0)
    per_img = s["objects"] / max(s["images"], 1)
    print(f"{s['split']:<8}{s['images']:>8}{s['objects']:>10}{n_head:>10}{n_person:>10}{per_img:>10.2f}")

split     images   objects      Head    Person   obj/img
--------------------------------------------------------
train      12231    115279     54310     60969      9.43
valid        899      8936      4145      4791      9.94
test         846      7953      3812      4141      9.40


In [7]:
# ============================================================
# EDA — plots
# ============================================================
train_stats = stats[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) class balance
counts = [train_stats["class_counts"].get(i, 0) for i in range(NUM_CLASSES)]
axes[0].bar(CLASS_NAMES, counts, color=["#4C72B0", "#DD8452"])
axes[0].set_title("Class balance (train)")
axes[0].set_ylabel("number of boxes")
for i, c in enumerate(counts):
    axes[0].text(i, c, f"{c:,}", ha="center", va="bottom")

# (b) objects per image — how crowded are the scenes?
axes[1].hist(train_stats["boxes_per_image"], bins=range(0, 31), color="#55A868")
axes[1].set_title("Objects per image (train)")
axes[1].set_xlabel("boxes in image")
axes[1].set_ylabel("images")

# (c) box area, log scale — the small-object story
for cls_id, name, colour in zip(range(NUM_CLASSES), CLASS_NAMES, ["#4C72B0", "#DD8452"]):
    a = [area for c, area in train_stats["areas"] if c == cls_id and area > 0]
    if a:
        axes[2].hist(np.log10(a), bins=50, alpha=0.6, label=name, color=colour)
axes[2].set_title("Box area (train), log scale")
axes[2].set_xlabel("log10(area as fraction of image)")
axes[2].set_ylabel("boxes")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "eda.png"), dpi=110)
plt.show()

# Quantify the small-object claim rather than just asserting it.
for cls_id, name in enumerate(CLASS_NAMES):
    a = np.array([area for c, area in train_stats["areas"] if c == cls_id])
    if len(a):
        tiny = float((a < 0.01).mean() * 100)  # <1% of image area
        print(f"{name:<8} median area={np.median(a) * 100:6.3f}% of image | {tiny:5.1f}% of boxes are <1% area")

NameError: name 'plt' is not defined

In [8]:
# ============================================================
# EDA — look at real images with their ground-truth boxes
# ============================================================
import cv2

COLOURS = [(66, 133, 244), (219, 132, 82)]  # Head=blue, Person=orange


def draw_boxes(img_rgb, boxes, labels, scores=None, thickness=2):
    """boxes: (N,4) absolute xyxy. Returns a copy — never mutates the input."""
    out = img_rgb.copy()
    for i, (box, lab) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = [int(v) for v in box]
        colour = COLOURS[int(lab) % len(COLOURS)]
        cv2.rectangle(out, (x1, y1), (x2, y2), colour, thickness)
        text = CLASS_NAMES[int(lab)]
        if scores is not None:
            text += f" {scores[i]:.2f}"
        cv2.putText(out, text, (x1, max(y1 - 5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1, cv2.LINE_AA)
    return out


def load_rgb(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"could not read image: {img_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


sample = random.Random(SEED).sample(TRAIN_PAIRS, min(6, len(TRAIN_PAIRS)))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (img_path, label_path) in zip(axes.ravel(), sample):
    img = load_rgb(img_path)
    h, w = img.shape[:2]
    gt = yolo_labels_to_xyxy(label_path, w, h)
    ax.imshow(draw_boxes(img, gt["boxes"], gt["labels"]))
    ax.set_title(f"{len(gt['boxes'])} boxes", fontsize=10)
    ax.axis("off")
plt.suptitle("Ground-truth samples (blue = Head, orange = Person)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "gt_samples.png"), dpi=110)
plt.show()

NameError: name 'plt' is not defined

In [9]:
# ============================================================
# Shared evaluation driver — every track goes through this
# ============================================================
RESULTS = {}


def build_ground_truth(pairs):
    """GT for the eval split, in absolute pixels of each ORIGINAL image."""
    gts = []
    for img_path, label_path in pairs:
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        gts.append(yolo_labels_to_xyxy(label_path, w, h))
    return gts


def score_track(track_name, predict_fn, pairs=None, notes=""):
    """Run predict_fn over the eval split and score it with the shared metric.

    predict_fn(img_rgb) -> dict with keys boxes (N,4 absolute xyxy), scores (N,), labels (N,)
    """
    pairs = pairs if pairs is not None else EVAL_PAIRS
    preds = []
    for img_path, _ in pairs:
        preds.append(predict_fn(load_rgb(img_path)))

    gts = build_ground_truth(pairs)
    res = evaluate_detections(preds, gts, CLASS_NAMES)
    res["notes"] = notes
    res["n_eval_images"] = len(pairs)
    RESULTS[track_name] = res

    print(f"\n=== {track_name} ===")
    print(f"  mAP50    : {res['mAP50']:.4f}")
    print(f"  mAP50-95 : {res['mAP50_95']:.4f}")
    for cls_name, ap in res["per_class"].items():
        print(f"  {cls_name:<8} AP50={ap['AP50']:.4f}  AP50-95={ap['AP50_95']:.4f}")
    return res


GROUND_TRUTH = build_ground_truth(EVAL_PAIRS)
print(f"ground truth prepared for {len(GROUND_TRUTH)} eval images")

ground truth prepared for 100 eval images


## 2. Track C — CNN trained from scratch

A deliberately small YOLOv1-lite detector: 4 conv blocks (Conv → BatchNorm → ReLU → Dropout2d →
MaxPool) take a 128×128 image down to an 8×8 grid, and a conv head predicts
`(tx, ty, tw, th, objectness, 2 class logits)` per cell.

**The key limitation, stated up front:** one box per grid cell. The EDA above shows ~9.4 objects per
image, so multiple objects routinely land in the same cell and only one survives into the training
target. This caps achievable recall no matter how long we train. The next cell measures that ceiling
exactly, before training anything — so Track C's result is explained rather than excused.

In [10]:
# ============================================================
# TRACK C — measure the encoding's recall ceiling BEFORE training
# ============================================================
# The target encoder keeps one box per cell. Any box landing in an already-taken
# cell is silently dropped and never contributes to the loss, so the model cannot
# learn to predict it. That puts a hard upper bound on recall that is a property
# of the encoding, not of training. Quantify it rather than hand-waving.

def grid_collision_rate(pairs, grid_size=8):
    kept = dropped = 0
    for _, label_path in pairs:
        occupied = set()
        for cls_id, xc, yc, w, h in read_yolo_labels(label_path):
            cell = (min(int(yc * grid_size), grid_size - 1),
                    min(int(xc * grid_size), grid_size - 1))
            if cell in occupied:
                dropped += 1
            else:
                occupied.add(cell)
                kept += 1
    return kept, dropped


for S in (8, 16, 32):
    kept, dropped = grid_collision_rate(TRAIN_PAIRS, S)
    total = kept + dropped
    print(f"{S:>2}x{S:<2} grid | kept {kept:>7,} | dropped {dropped:>7,} "
          f"| {dropped / total * 100:5.1f}% of boxes lost | recall ceiling {kept / total * 100:5.1f}%")

print("\nThe 8x8 row is the encoding Track C actually uses. Finer grids recover most of the loss,")
print("which is why 'more epochs' cannot fix Track C but a denser grid / more anchors would.")

 8x8  grid | kept  81,405 | dropped  33,874 |  29.4% of boxes lost | recall ceiling  70.6%
16x16 grid | kept 103,062 | dropped  12,217 |  10.6% of boxes lost | recall ceiling  89.4%
32x32 grid | kept 111,817 | dropped   3,462 |   3.0% of boxes lost | recall ceiling  97.0%

The 8x8 row is the encoding Track C actually uses. Finer grids recover most of the loss,
which is why 'more epochs' cannot fix Track C but a denser grid / more anchors would.


In [11]:
# ============================================================
# TRACK C — dataset, model, loss
# ============================================================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

IMG_SIZE, GRID_SIZE = 128, 8


class HeadPersonGridDataset(Dataset):
    """YOLOv1-style targets: one box per grid cell, encoded at (5+C, S, S)."""

    def __init__(self, pairs, subset_size=None, img_size=IMG_SIZE, grid_size=GRID_SIZE, seed=SEED):
        pairs = list(pairs)
        random.Random(seed).shuffle(pairs)
        self.pairs = pairs[:subset_size] if subset_size else pairs
        self.img_size, self.grid_size = img_size, grid_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, label_path = self.pairs[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        S = self.grid_size
        target = torch.zeros((5 + NUM_CLASSES, S, S), dtype=torch.float32)
        for cls_id, xc, yc, w, h in read_yolo_labels(label_path):
            gi, gj = min(int(xc * S), S - 1), min(int(yc * S), S - 1)
            if target[4, gj, gi] == 1.0:
                continue  # cell already taken — the one-box-per-cell limitation
            target[0, gj, gi] = xc * S - gi
            target[1, gj, gi] = yc * S - gj
            target[2, gj, gi] = w
            target[3, gj, gi] = h
            target[4, gj, gi] = 1.0
            target[5 + cls_id, gj, gi] = 1.0
        return img_t, target


def conv_block(in_ch, out_ch, dropout=0.1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Dropout2d(dropout),
        nn.MaxPool2d(2),
    )


class ScratchDetector(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, grid_size=GRID_SIZE, dropout=0.1):
        super().__init__()
        self.grid_size = grid_size
        self.backbone = nn.Sequential(          # 128 -> 64 -> 32 -> 16 -> 8
            conv_block(3, 16, dropout), conv_block(16, 32, dropout),
            conv_block(32, 64, dropout), conv_block(64, 128, dropout),
        )
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.Conv2d(128, 5 + num_classes, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


LAMBDA_COORD, LAMBDA_NOOBJ = 5.0, 0.5


class ScratchDetectorLoss(nn.Module):
    """YOLOv1 sum-squared-error. LAMBDA_NOOBJ down-weights the ~64x more common
    empty cells so they don't drown out the objectness signal."""

    def forward(self, pred, target):
        p_xy, p_wh = torch.sigmoid(pred[:, 0:2]), torch.sigmoid(pred[:, 2:4])
        p_obj, p_cls = torch.sigmoid(pred[:, 4]), pred[:, 5:]
        t_xy, t_wh = target[:, 0:2], target[:, 2:4]
        t_obj, t_cls = target[:, 4], target[:, 5:]

        obj = t_obj.unsqueeze(1)
        coord = LAMBDA_COORD * (((p_xy - t_xy) ** 2 * obj).sum() + ((p_wh - t_wh) ** 2 * obj).sum())
        obj_l = ((p_obj - t_obj) ** 2 * t_obj).sum()
        noobj = LAMBDA_NOOBJ * (p_obj ** 2 * (1.0 - t_obj)).sum()
        cls_l = ((p_cls.sigmoid() - t_cls) ** 2 * obj).sum()
        return (coord + obj_l + noobj + cls_l) / pred.shape[0]


_m = ScratchDetector()
print(f"ScratchDetector params: {sum(p.numel() for p in _m.parameters()):,}")
print("output shape:", tuple(_m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape))

ModuleNotFoundError: No module named 'torch'

In [12]:
# ============================================================
# TRACK C — train
# ============================================================
import time

C_SUBSET = 800 if QUICK_MODE else 6000
C_EPOCHS = 5 if QUICK_MODE else 40
C_BATCH, C_LR = 16, 1e-3

# num_workers=0 on non-Linux: worker processes are slow/fragile on macOS and Windows.
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0

train_ds = HeadPersonGridDataset(TRAIN_PAIRS, C_SUBSET, seed=SEED)
val_ds = HeadPersonGridDataset(VALID_PAIRS, max(100, C_SUBSET // 4), seed=SEED + 1)
train_loader = DataLoader(train_ds, batch_size=C_BATCH, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=C_BATCH, shuffle=False, num_workers=NUM_WORKERS)
print(f"Track C — train {len(train_ds)} | val {len(val_ds)} | epochs {C_EPOCHS}")

model_c = ScratchDetector().to(DEVICE)
criterion = ScratchDetectorLoss()
optimizer = torch.optim.Adam(model_c.parameters(), lr=C_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=C_EPOCHS)

TRACK_C_OUT = Path(OUTPUT_ROOT) / "track_c"
TRACK_C_OUT.mkdir(parents=True, exist_ok=True)
best_val, history = float("inf"), {"train": [], "val": []}

for epoch in range(C_EPOCHS):
    t0 = time.time()

    model_c.train()
    running = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_c(imgs), targets)
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    train_loss = running / len(train_ds)

    model_c.eval()
    running = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            running += criterion(model_c(imgs), targets).item() * imgs.size(0)
    val_loss = running / len(val_ds)

    history["train"].append(train_loss)
    history["val"].append(val_loss)
    scheduler.step()

    # Keep the best-on-validation weights, not merely the last epoch's.
    flag = ""
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model_c.state_dict(), TRACK_C_OUT / "scratch_detector.pt")
        flag = "  <- best, saved"

    print(f"epoch {epoch+1:>3}/{C_EPOCHS}  train={train_loss:8.4f}  val={val_loss:8.4f}"
          f"  ({time.time()-t0:.1f}s){flag}")

model_c.load_state_dict(torch.load(TRACK_C_OUT / "scratch_detector.pt", map_location=DEVICE))
model_c.eval()
print(f"\nbest val loss: {best_val:.4f} -> {TRACK_C_OUT / 'scratch_detector.pt'}")

plt.figure(figsize=(6, 4))
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Track C — training curve"); plt.grid(alpha=0.3)
plt.savefig(TRACK_C_OUT / "loss_curve.png", dpi=110)
plt.show()

NameError: name 'HeadPersonGridDataset' is not defined

In [13]:
# ============================================================
# TRACK C — decode, NMS, and score on the shared metric
# ============================================================
def decode_predictions(pred, grid_size=GRID_SIZE, conf_thresh=0.25):
    """Raw (5+C,S,S) logits -> [x1,y1,x2,y2,score,cls] in NORMALISED [0,1] coords.
    Vectorised so scoring the full test split stays quick."""
    S = grid_size
    xy = torch.sigmoid(pred[0:2])
    wh = torch.sigmoid(pred[2:4])
    obj = torch.sigmoid(pred[4])
    cls = torch.softmax(pred[5:], dim=0)

    cls_score, cls_id = cls.max(dim=0)
    score = obj * cls_score
    keep = score >= conf_thresh
    if not keep.any():
        return np.zeros((0, 6), dtype=np.float32)

    gj, gi = torch.nonzero(keep, as_tuple=True)
    xc = (gi.float() + xy[0][keep]) / S
    yc = (gj.float() + xy[1][keep]) / S
    w, h = wh[0][keep], wh[1][keep]

    out = torch.stack([
        xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2,
        score[keep], cls_id[keep].float(),
    ], dim=1)
    return out.cpu().numpy().astype(np.float32)


def nms(boxes, iou_thresh=0.45):
    """Greedy per-class NMS on [x1,y1,x2,y2,score,cls] rows."""
    if len(boxes) == 0:
        return boxes
    keep = []
    for cls_id in np.unique(boxes[:, 5]):
        cls_boxes = boxes[boxes[:, 5] == cls_id]
        cls_boxes = cls_boxes[np.argsort(-cls_boxes[:, 4])]
        while len(cls_boxes):
            best, cls_boxes = cls_boxes[0], cls_boxes[1:]
            keep.append(best)
            if len(cls_boxes):
                ious = iou_matrix(best[None, :4], cls_boxes[:, :4])[0]
                cls_boxes = cls_boxes[ious < iou_thresh]
    return np.array(keep, dtype=np.float32)


def predict_track_c(img_rgb, conf_thresh=0.25):
    h, w = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    t = torch.from_numpy(resized).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE) / 255.0
    with torch.no_grad():
        raw = model_c(t)[0]

    boxes = nms(decode_predictions(raw, conf_thresh=conf_thresh))
    if len(boxes) == 0:
        return {"boxes": np.zeros((0, 4), np.float32),
                "scores": np.zeros(0, np.float32), "labels": np.zeros(0, int)}

    # Normalised -> absolute pixels of the ORIGINAL image, so GT and preds agree.
    abs_boxes = boxes[:, :4] * np.array([w, h, w, h], dtype=np.float32)
    return {"boxes": abs_boxes, "scores": boxes[:, 4], "labels": boxes[:, 5].astype(int)}


score_track("Track C — CNN from scratch", predict_track_c,
            notes=f"{IMG_SIZE}px input, {GRID_SIZE}x{GRID_SIZE} grid, 1 box/cell, "
                  f"{len(train_ds)} train images, {C_EPOCHS} epochs")

NameError: name 'GRID_SIZE' is not defined

## 3. Track A — YOLO (Ultralytics), pretrained → fine-tuned

The pretrained COCO backbone already knows what a person looks like, so fine-tuning mostly teaches
it the new `Head` class and this dataset's box conventions. This is the strongest
accuracy-per-unit-effort option and should win.

In [14]:
# ============================================================
# TRACK A — fine-tune
# ============================================================
from ultralytics import YOLO

A_EPOCHS = 3 if QUICK_MODE else 50
A_IMGSZ = 640
A_BATCH = 16 if DEVICE.type == "cuda" else 8

# yolo26n needs a recent Ultralytics; fall back to yolov8n (what this dataset was
# exported for) so the notebook runs on any version.
MODEL_CANDIDATES = ["yolo26n.pt", "yolo11n.pt", "yolov8n.pt"]
model_a, chosen = None, None
for name in MODEL_CANDIDATES:
    try:
        model_a = YOLO(name)
        chosen = name
        break
    except Exception as exc:
        print(f"  {name} unavailable ({type(exc).__name__}) -> trying next")

assert model_a is not None, f"none of {MODEL_CANDIDATES} could be loaded"
print("Track A base checkpoint:", chosen)

TRACK_A_OUT = Path(OUTPUT_ROOT) / "track_a"
model_a.train(
    data=DATA_YAML,
    epochs=A_EPOCHS,
    imgsz=A_IMGSZ,
    batch=A_BATCH,
    project=str(TRACK_A_OUT),
    name="finetune",
    exist_ok=True,
    seed=SEED,
    workers=NUM_WORKERS,
    device=0 if DEVICE.type == "cuda" else DEVICE.type,
)
print("Track A training done")

ModuleNotFoundError: No module named 'ultralytics'

In [15]:
# ============================================================
# TRACK A — score on the shared metric
# ============================================================
A_CONF = 0.25

def predict_track_a(img_rgb):
    # Ultralytics wants BGR when handed a numpy array.
    r = model_a.predict(img_rgb[:, :, ::-1], conf=A_CONF, verbose=False, device=model_a.device)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return {"boxes": np.zeros((0, 4), np.float32),
                "scores": np.zeros(0, np.float32), "labels": np.zeros(0, int)}
    return {
        "boxes": r.boxes.xyxy.cpu().numpy().astype(np.float32),
        "scores": r.boxes.conf.cpu().numpy().astype(np.float32),
        "labels": r.boxes.cls.cpu().numpy().astype(int),
    }


score_track("Track A — YOLO fine-tuned", predict_track_a,
            notes=f"{chosen}, {A_IMGSZ}px, {A_EPOCHS} epochs, full train split")

# Cross-check against Ultralytics' own validator. The two should be close; a large
# gap means the shared metric is being fed boxes in the wrong space.
try:
    m = model_a.val(data=DATA_YAML, split="test" if os.path.isdir(os.path.join(DATASET_ROOT, "test")) else "val",
                    verbose=False)
    print(f"\ncross-check — Ultralytics mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")
    print(f"              shared metric mAP50={RESULTS['Track A — YOLO fine-tuned']['mAP50']:.4f}"
          f"  (evaluated on {EVAL_N} images, conf={A_CONF})")
except Exception as exc:
    print("Ultralytics cross-check skipped:", exc)

NameError: name 'chosen' is not defined

## 4. Track B — RF-DETR (transformer), pretrained → fine-tuned

RF-DETR reads the YOLO-format folder directly (no COCO conversion). It is a DETR-family
transformer: set prediction with no anchors and no NMS. Worth comparing against Track A to see
whether the transformer's global attention helps on the crowded scenes the EDA revealed.

In [16]:
# ============================================================
# TRACK B — fine-tune
# ============================================================
B_EPOCHS = 2 if QUICK_MODE else 40
B_BATCH = 8 if DEVICE.type == "cuda" else 2

TRACK_B_OUT = Path(OUTPUT_ROOT) / "track_b" / "finetune"
TRACK_B_OUT.parent.mkdir(parents=True, exist_ok=True)

track_b_ok = False
try:
    from rfdetr import RFDETRNano

    model_b = RFDETRNano()
    model_b.train(
        dataset_dir=DATASET_ROOT,
        epochs=B_EPOCHS,
        batch_size=B_BATCH,
        lr=1e-4,
        output_dir=str(TRACK_B_OUT),
    )
    track_b_ok = True
    print("Track B training done")
except Exception as exc:
    # RF-DETR's training path assumes CUDA; on CPU/MPS it typically fails. Don't
    # let that take down the rest of the notebook.
    print(f"Track B skipped: {type(exc).__name__}: {exc}")
    print("RF-DETR needs a CUDA GPU — run this section on Colab/Kaggle with a T4.")

NameError: name 'DEVICE' is not defined

In [17]:
# ============================================================
# TRACK B — score on the shared metric
# ============================================================
if track_b_ok:
    from PIL import Image

    ckpt = TRACK_B_OUT / "checkpoint_best_total.pth"
    model_b_eval = RFDETRNano.from_checkpoint(str(ckpt)) if ckpt.exists() else model_b

    def predict_track_b(img_rgb, threshold=0.3):
        det = model_b_eval.predict(Image.fromarray(img_rgb), threshold=threshold)
        if len(det) == 0:
            return {"boxes": np.zeros((0, 4), np.float32),
                    "scores": np.zeros(0, np.float32), "labels": np.zeros(0, int)}
        labels = np.asarray(det.class_id, dtype=int)
        # RF-DETR can emit 1-indexed class ids; realign to our 0-indexed names.
        if labels.size and labels.min() >= 1 and labels.max() >= NUM_CLASSES:
            labels = labels - 1
        return {
            "boxes": np.asarray(det.xyxy, dtype=np.float32),
            "scores": np.asarray(det.confidence, dtype=np.float32),
            "labels": labels,
        }

    score_track("Track B — RF-DETR fine-tuned", predict_track_b,
                notes=f"RFDETRNano, {B_EPOCHS} epochs, full train split")
else:
    print("Track B was not trained — nothing to score.")

NameError: name 'track_b_ok' is not defined

## 5. Results — all three tracks, one metric

Every number below came from the same `evaluate_detections` call on the same test images, so the
ranking is meaningful.

In [18]:
# ============================================================
# Comparison table + chart + saved results.json
# ============================================================
import pandas as pd

rows = []
for name, r in RESULTS.items():
    row = {
        "Track": name,
        "mAP50": round(r["mAP50"], 4),
        "mAP50-95": round(r["mAP50_95"], 4),
        "eval imgs": r["n_eval_images"],
    }
    for cls_name, ap in r["per_class"].items():
        row[f"AP50 ({cls_name})"] = round(ap["AP50"], 4)
    rows.append(row)

df = pd.DataFrame(rows).sort_values("mAP50", ascending=False).reset_index(drop=True)
display(df)

if len(df):
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(df))
    ax.bar(x - 0.2, df["mAP50"], 0.4, label="mAP50", color="#4C72B0")
    ax.bar(x + 0.2, df["mAP50-95"], 0.4, label="mAP50-95", color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace(" — ", "\n") for t in df["Track"]], fontsize=9)
    ax.set_ylabel("score"); ax.set_ylim(0, 1)
    ax.set_title("Track comparison — identical metric, identical test images")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, "comparison.png"), dpi=110)
    plt.show()

summary_path = os.path.join(OUTPUT_ROOT, "results.json")
with open(summary_path, "w") as f:
    json.dump({
        "quick_mode": QUICK_MODE,
        "dataset_root": DATASET_ROOT,
        "eval_images": len(EVAL_PAIRS),
        "device": str(DEVICE),
        "results": RESULTS,
    }, f, indent=2)
print("saved ->", summary_path)
df.to_csv(os.path.join(OUTPUT_ROOT, "results.csv"), index=False)

KeyError: 'mAP50'

In [ ]:
# ============================================================
# Qualitative — same images, every track, side by side
# ============================================================
predictors = {"Ground truth": None}
if "Track C — CNN from scratch" in RESULTS:
    predictors["Track C (scratch)"] = predict_track_c
if "Track A — YOLO fine-tuned" in RESULTS:
    predictors["Track A (YOLO)"] = predict_track_a
if track_b_ok and "Track B — RF-DETR fine-tuned" in RESULTS:
    predictors["Track B (RF-DETR)"] = predict_track_b

n_show = min(4, len(EVAL_PAIRS))
show_pairs = random.Random(SEED).sample(EVAL_PAIRS, n_show)

fig, axes = plt.subplots(n_show, len(predictors), figsize=(4.5 * len(predictors), 4 * n_show))
axes = np.atleast_2d(axes)

for r, (img_path, label_path) in enumerate(show_pairs):
    img = load_rgb(img_path)
    h, w = img.shape[:2]
    for c, (name, fn) in enumerate(predictors.items()):
        ax = axes[r, c]
        if fn is None:
            gt = yolo_labels_to_xyxy(label_path, w, h)
            ax.imshow(draw_boxes(img, gt["boxes"], gt["labels"]))
            n = len(gt["boxes"])
        else:
            p = fn(img)
            ax.imshow(draw_boxes(img, p["boxes"], p["labels"], p["scores"]))
            n = len(p["boxes"])
        if r == 0:
            ax.set_title(name, fontsize=12)
        ax.set_xlabel(f"{n} boxes", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Qualitative comparison (blue = Head, orange = Person)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "qualitative.png"), dpi=110)
plt.show()

## 6. Conclusions

Fill these in from **your** run — the numbers depend on `QUICK_MODE`, epochs, and the GPU you used.

**Expected ordering:** Track A ≥ Track B ≫ Track C.

**Why Track C loses, concretely:**

1. **One box per grid cell.** The EDA histogram shows many images with 10+ objects. On an 8×8 grid,
   crowded regions put several heads in one cell and all but one is silently dropped from the
   training target — a hard recall ceiling that no amount of training removes.
2. **128×128 input.** The box-area histogram shows most `Head` boxes cover well under 1% of the
   image. At 128px those are a handful of pixels; Tracks A and B see 640px.
3. **No pretraining.** A and B start from checkpoints that already have general object features.
   C learns edges from zero on a few thousand images.

**How you would close the gap** (in rough order of payoff): raise input resolution to 416px, predict
multiple anchors per cell, add multi-scale (FPN-style) prediction heads, then add augmentation.

**Head vs Person AP** is worth a sentence in the report too — if `Head` AP trails `Person` AP for
every track, that is the small-object effect showing up consistently rather than a quirk of one model.